In [5]:
from dotenv import load_dotenv
import os

load_dotenv()
from huggingface_hub import login
login(token=os.getenv("HF_TOKEN"))

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to C:\Users\NISHIT\.cache\huggingface\token
Login successful


In [6]:
from transformers import pipeline
ner = pipeline("ner", aggregation_strategy='simple', device=0)

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision f2482bf (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
c:\Users\NISHIT\AppData\Local\Pr

In [7]:
import pickle
# conll 2003
!wget -nc https://lazyprogrammer.me/course_files/nlp/ner_train.pkl
!wget -nc https://lazyprogrammer.me/course_files/nlp/ner_test.pkl

--2026-05-17 13:51:43--  https://lazyprogrammer.me/course_files/nlp/ner_train.pkl
Resolving lazyprogrammer.me (lazyprogrammer.me)... 2606:4700:3031::6815:17d2, 2606:4700:3030::ac43:d5a6, 104.21.23.210, ...
Connecting to lazyprogrammer.me (lazyprogrammer.me)|2606:4700:3031::6815:17d2|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4757208 (4.5M) [application/octet-stream]
Saving to: 'ner_train.pkl'

     0K .......... .......... .......... .......... ..........  1%  346K 13s
    50K .......... .......... .......... .......... ..........  2%  814K 9s
   100K .......... .......... .......... .......... ..........  3% 16.3M 6s
   150K .......... .......... .......... .......... ..........  4% 9.89M 5s
   200K .......... .......... .......... .......... ..........  5%  867K 5s
   250K .......... .......... .......... .......... ..........  6% 13.7M 4s
   300K .......... .......... .......... .......... ..........  7% 30.0M 3s
   350K .......... .......... .........

In [8]:
with open('ner_train.pkl', 'rb') as f:
  corpus_train = pickle.load(f)

with open('ner_test.pkl', 'rb') as f:
  corpus_test = pickle.load(f)

In [9]:
inputs = []
targets = []

for sentence_tag_pairs in corpus_test:
  tokens = []
  target = []
  for token, tag in sentence_tag_pairs:
    tokens.append(token)
    target.append(tag)
  inputs.append(tokens)
  targets.append(target)

In [10]:
inputs[9]

['He',
 'was',
 'well',
 'backed',
 'by',
 'England',
 'hopeful',
 'Mark',
 'Butcher',
 'who',
 'made',
 '70',
 'as',
 'Surrey',
 'closed',
 'on',
 '429',
 'for',
 'seven',
 ',',
 'a',
 'lead',
 'of',
 '234',
 '.']

In [12]:
from nltk.tokenize.treebank import TreebankWordDetokenizer
detokenizer = TreebankWordDetokenizer()
detokenizer.detokenize(inputs[3])

'After bowling Somerset out for 83 on the opening morning at Grace Road, Leicestershire extended their first innings by 94 runs before being bowled out for 296 with England discard Andy Caddick taking three for 83.'

In [13]:
ner(detokenizer.detokenize(inputs[3]))


[{'entity_group': 'ORG',
  'score': 0.9988477,
  'word': 'Somerset',
  'start': 14,
  'end': 22},
 {'entity_group': 'LOC',
  'score': 0.99434716,
  'word': 'Grace Road',
  'start': 60,
  'end': 70},
 {'entity_group': 'ORG',
  'score': 0.99868876,
  'word': 'Leicestershire',
  'start': 72,
  'end': 86},
 {'entity_group': 'LOC',
  'score': 0.9996426,
  'word': 'England',
  'start': 164,
  'end': 171},
 {'entity_group': 'PER',
  'score': 0.9896603,
  'word': 'Andy Caddick',
  'start': 180,
  'end': 192}]

In [14]:
def compute_prediction(tokens, input_, ner_result):
  # map hugging face ner result to list of tags for later performance assessment
  # tokens is the original tokenized sentence
  # input_ is the detokenized string

  predicted_tags = []
  state = 'O' # keep track of state, so if O --> B, if B --> I, if I --> I
  current_index = 0

  # keep track of last group since the group may change
  # between consecutive entities
  # e.g. we want B-MISC -> B-PER -> I-PER
  # not          B-MISC -> I-PER -> I-PER
  last_group = None

  for token in tokens:
    # find the token in the input_ (should be at or near the start)
    index = input_.find(token)
    assert(index >= 0)
    current_index += index # where we are currently pointing to

    # print(token, current_index) # debug

    # check if this index belongs to an entity and assign label
    tag = 'O'
    for entity in ner_result:
      group = entity['entity_group']
      if current_index >= entity['start'] and current_index < entity['end']:
        # then this token belongs to an entity
        if state == 'O':
          state = 'B'
        elif last_group != group:
          state = 'B'
        else:
          state = 'I'
        tag = f"{state}-{group}"
        last_group = group
        break
    if tag == 'O':
      # reset the state
      state = 'O'
      last_group = None
    predicted_tags.append(tag)

    # remove the token from input_
    input_ = input_[index + len(token):]

    # update current_index
    current_index += len(token)

  # sanity check
  # print("len(predicted_tags)", len(predicted_tags))
  # print("len(tokens)", len(tokens))
  assert(len(predicted_tags) == len(tokens))
  return predicted_tags

In [15]:
input_ = detokenizer.detokenize(inputs[9])
ner_result = ner(input_)
ptags = compute_prediction(inputs[9], input_, ner_result)
# TMP
input2 = detokenizer.detokenize(inputs[11])
ner_result2 = ner(input2)
ptags2 = compute_prediction(inputs[11], input2, ner_result2)
ptags2

['B-MISC',
 'B-PER',
 'I-PER',
 'O',
 'O',
 'O',
 'O',
 'O',
 'B-PER',
 'I-PER',
 'O',
 'O',
 'O',
 'O',
 'B-PER',
 'I-PER',
 'O',
 'O',
 'O',
 'O',
 'B-ORG',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O',
 'O']

In [16]:
from sklearn.metrics import accuracy_score, f1_score
accuracy_score(targets[9], ptags)

1.0

In [17]:
for targ, pred in zip(targets[9], ptags):
  print(targ, pred)

O O
O O
O O
O O
O O
B-LOC B-LOC
O O
B-PER B-PER
I-PER I-PER
O O
O O
O O
O O
B-ORG B-ORG
O O
O O
O O
O O
O O
O O
O O
O O
O O
O O
O O


In [18]:
for targ, pred in zip(targets[9], ptags):
  print(targ, pred)

O O
O O
O O
O O
O O
B-LOC B-LOC
O O
B-PER B-PER
I-PER I-PER
O O
O O
O O
O O
B-ORG B-ORG
O O
O O
O O
O O
O O
O O
O O
O O
O O
O O
O O


In [20]:
# get detokenized inputs to pass into ner model
detok_inputs = []
for tokens in inputs:
  text = detokenizer.detokenize(tokens)
  detok_inputs.append(text)

In [21]:
ner_results = ner(detok_inputs)
predictions = []
for tokens, text, ner_result in zip(inputs, detok_inputs, ner_results):
  pred = compute_prediction(tokens, text, ner_result)
  predictions.append(pred)

In [22]:
# https://stackoverflow.com/questions/11264684/flatten-list-of-lists
def flatten(list_of_lists):
  flattened = [val for sublist in list_of_lists for val in sublist]
  return flattened
# flatten targets and predictions
flat_predictions = flatten(predictions)
flat_targets = flatten(targets)

In [23]:
accuracy_score(flat_targets, flat_predictions)


0.9920892614676191

In [26]:
f1_score(flat_targets, flat_predictions, average='macro')


0.9571181287370304